# Model Monitoring & Drift Detection: Data Drift, Concept Drift & Telemetry

Production ML models degrade over time as real-world environments shift:
1. **Data Drift (Covariate Shift)**: Detecting when input distributions $P(X)$ change using Kolmogorov-Smirnov (KS) tests and Population Stability Index (PSI).
2. **Concept Drift**: Detecting when the true relationship $P(Y | X)$ shifts, causing model accuracy/loss to degrade.
3. **Earth Mover's (Wasserstein) Distance**: Quantifying the physical magnitude of distribution shifts.
4. **MLOps Observability & Prometheus Export**: Streaming telemetry (latency percentiles, drift alerts) to monitoring dashboards.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('07-mlops/monitoring-drift/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
import drift_detector
print('drift_detector loaded')

## 1. Statistical Data Drift: Kolmogorov-Smirnov Test & PSI
Comparing live incoming production feature distributions against baseline reference distributions.

In [ ]:
import numpy as np
from drift_detector import kolmogorov_smirnov_2sample, population_stability_index, wasserstein_distance_1d

np.random.seed(42)

# Golden training reference distribution: Transaction Amount (normal with mean $50)
ref_amounts = np.random.normal(50.0, 15.0, 1000).tolist()

# 1. Normal production batch (unchanged)
prod_normal = np.random.normal(50.5, 14.8, 500).tolist()
ks_norm, p_norm = kolmogorov_smirnov_2sample(ref_amounts, prod_normal)
psi_norm = population_stability_index(ref_amounts, prod_normal)
w_norm = wasserstein_distance_1d(ref_amounts, prod_normal)

print(f"Normal Batch  -> KS Stat: {ks_norm:.4f} (p={p_norm:.4f}) | PSI: {psi_norm:.4f} | W-Dist: {w_norm:.4f}")

# 2. Shifted production batch (holiday shopping spike: mean $85)
prod_shifted = np.random.normal(85.0, 25.0, 500).tolist()
ks_shift, p_shift = kolmogorov_smirnov_2sample(ref_amounts, prod_shifted)
psi_shift = population_stability_index(ref_amounts, prod_shifted)
w_shift = wasserstein_distance_1d(ref_amounts, prod_shifted)

print(f"Shifted Batch -> KS Stat: {ks_shift:.4f} (p={p_shift:.4e}) | PSI: {psi_shift:.4f} | W-Dist: {w_shift:.4f}")
print("PSI Decision Rule: PSI >= 0.25 flags SIGNIFICANT DRIFT requiring model retraining!")

## 2. Multi-Feature Data Drift Pipeline
Automated monitoring report across all input features.

In [ ]:
from drift_detector import DataDriftDetector

baseline_dataset = {
    "credit_score": np.random.normal(700, 50, 800).tolist(),
    "debt_to_income": np.random.exponential(0.3, 800).tolist(),
    "monthly_income": np.random.normal(6000, 1500, 800).tolist()
}

detector = DataDriftDetector(baseline_dataset, p_value_threshold=0.01, psi_threshold=0.25)

# Production traffic where debt_to_income has drifted due to inflation
live_batch = {
    "credit_score": np.random.normal(698, 52, 300).tolist(),             # Stable
    "debt_to_income": np.random.exponential(0.65, 300).tolist(),          # DRIFTED!
    "monthly_income": np.random.normal(6050, 1480, 300).tolist()          # Stable
}

report = detector.evaluate_batch(live_batch)
print("--- Data Drift Report ---")
for feat, summary in report.items():
    flag = "DRIFT DETECTED" if summary.is_drifted else "STABLE"
    print(f"[{flag:14s}] Feature: {feat:16s} | PSI: {summary.psi:.4f} | KS p-val: {summary.ks_pvalue:.4f}")

## 3. Concept Drift & Model Performance Degradation
Detecting when ground-truth labels diverge from model expectations.

In [ ]:
from drift_detector import ConceptDriftDetector

# Production baseline accuracy: 92%
concept_mon = ConceptDriftDetector(baseline_metric=0.92, threshold_drop=0.12, window_size=50)

# Simulate 40 healthy predictions
for _ in range(40):
    concept_mon.log_prediction(1.0)

# Sudden macroeconomic event degrades performance
alert_triggered = None
for step in range(50):
    alert = concept_mon.log_prediction(0.40)  # low accuracy
    if alert:
        alert_triggered = alert
        print(f"Alert fired at step {step}: {alert}")
        break

## 4. Observability & Prometheus Metrics Telemetry
Exporting real-time serving latency percentiles and drift metrics for Grafana alerting.

In [ ]:
from drift_detector import PrometheusMetricsExporter

exporter = PrometheusMetricsExporter(model_name="credit_risk_xgb", model_version="v2.0.0")

# Record requests and alerts
for latency in [14.2, 16.5, 15.0, 19.8, 22.4, 45.1, 15.6]:
    exporter.record_request(latency_ms=latency, has_drift=False)
exporter.record_request(latency_ms=62.0, has_drift=True)

metrics_payload = exporter.export_metrics()
print("--- Prometheus Text Exposition Format ---")
print(metrics_payload)